# Assignment 1: ConvNet Classifier

In this notebook, you will implement a ConvNet classifier to perform Fashion MNIST image classification.

The model has the following backbone.

- The input $x \sim\times 28\times 28$
- $x = \mathrm{relu}(\mathrm{conv}(x, \dots))\sim 64\times 28\times 28$
- $x = \mathrm{maxpool}(x, \dots)\sim 64\times 14\times 14$
- $x = \mathrm{relu}(\mathrm{conv}(x, \dots))\sim 128\times 14\times 14$
- $x = \mathrm{maxpool}(x, \dots)\sim 128\times 7\times 7$
- $x = \mathrm{relu}(\mathrm{conv}(x, \dots))\sim 512\times 7\times 7$

The output from the backbone is a $7\times 7$ feature map with 512 channels.  The classifier head performs 1x1 convolution to map to 10 channels, and uses global average pooling compute the logits.

In [ ]:
! pip install -q git+https://github.com/ontariotechu-csci4052u/csci4052u
import csci4052u
print("CSCI 4052U version:", csci4052u.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 70.2 MB/s eta 0:00:00
CSCI 4052U version: 2026-03-19


## Dataset

In [ ]:
# @title Load the Lightning Data Module.
frac_str = "1.0" # @param ["0.1","0.2","1.0"]
fraction = float(frac_str)

data = csci4052u.data.FashionMNISTDataModule(fraction=fraction)
data.prepare_data()

100%|██████████| 26.4M/26.4M [00:02<00:00, 11.6MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 171kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.11MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 24.0MB/s]


## Lightning Module

In [ ]:
from lightning import LightningModule
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchmetrics

class ConvNet(LightningModule):
    def __init__(self, num_classes=10, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()

        # Backbone: conv + ReLU + maxpool layers
        # Input: 1x28x28
        self.backbone = nn.Sequential(
            # Block 1: 1 -> 64 (spatial: 28x28 -> 14x14)
            nn.Conv2d(in_channels=1, out_channels=64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 2: 64 -> 128 (spatial: 14x14 -> 7x7)
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 3: 128 -> 512 (spatial: 7x7 -> 3x3)
            nn.Conv2d(in_channels=128, out_channels=512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # Classifier: 1x1 conv to num_classes, then global average pooling
        self.classifier = nn.Sequential(
            # 1x1 conv maps 512 channels down to 10 (num_classes)
            nn.Conv2d(in_channels=512, out_channels=num_classes, kernel_size=1, stride=1, padding=0),
            # Global Average Pooling squeezes spatial dimensions (H, W) to 1x1
            nn.AdaptiveAvgPool2d((1, 1)),
            # Flatten from (Batch, num_classes, 1, 1) -> (Batch, num_classes)
            nn.Flatten(start_dim=1)
        )

        # Metrics
        self.train_acc = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.val_acc = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.test_acc = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)

    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        return logits

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)

        self.train_acc(logits, y)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("train_acc", self.train_acc, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)

        self.val_acc(logits, y)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_acc", self.val_acc, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)

        self.test_acc(logits, y)
        self.log("test_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("test_acc", self.test_acc, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

## Training

In [ ]:
csci4052u.training.train(
    model_name="ConvNet",
    model=ConvNet(),
    datamodule=data,
    max_epochs=10,
    monitor_metric="val_loss",
)

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ backbone   │ Sequential         │  664 K │ train │     0 │
│ 1 │ classifier │ Sequential         │  5.1 K │ train │     0 │
│ 2 │ train_acc  │ MulticlassAccuracy │      0 │ train │     0 │
│ 3 │ val_acc    │ MulticlassAccuracy │      0 │ train │     0 │
│ 4 │ test_acc   │ MulticlassAccuracy │      0 │ train │     0 │
└───┴────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 669 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 669 K                                                                                                
Total estimated model params size (MB): 2.680                                                                      
Modules in train mode: 17                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.9210000038146973     │
│         test_loss         │    0.23237743973731995    │
└───────────────────────────┴───────────────────────────┘

In [ ]:
csci4052u.report.summarize(all_versions=True)

                                        Training Summary                                         
┏━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┓
┃ Model   ┃ Version   ┃ Train Acc ┃ Val Acc ┃ Test Acc ┃ Trainable ┃ Frozen ┃ Epochs ┃ Duration ┃
┡━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━┩
│ ConvNet │ version_0 │    0.9260 │  0.9179 │   0.9143 │    670.0K │      0 │     10 │   5m 17s │
│ ConvNet │ version_1 │    0.9501 │  0.9257 │   0.9210 │    670.0K │      0 │     10 │   5m 18s │
└─────────┴───────────┴───────────┴─────────┴──────────┴───────────┴────────┴────────┴──────────┘